# 03 - Measurement

**Prerequisites:** Notebooks 01-02. Familiarity with qubits, gates, and statevector simulation.

Measurement is where the quantum world meets the classical world. When we measure a qubit, we force it to reveal a definite classical value -- either 0 or 1 -- but this act **irreversibly destroys** the quantum superposition.

In this notebook we explore:
- **Computational basis measurement** and the Born rule
- **Measurement collapse** -- the common misconception that "measurement just reads the state"
- **Mid-circuit measurement** with dynamic circuits and classical feed-forward
- **Partial measurement** on entangled states
- **The no-cloning theorem** -- why you can't copy an unknown quantum state

By the end of this notebook you will be able to:

1. **Explain** the Born rule and predict measurement outcome probabilities.
2. **Demonstrate** that measurement collapses superposition using mid-circuit measurement.
3. **Describe** the no-cloning theorem and show why CNOT does not clone a superposition.
4. **Implement** dynamic circuits with classically conditioned gates.

In [1]:
import (
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Computational Basis Measurement

The simplest measurement is in the **computational basis** $\{|0\rangle, |1\rangle\}$. The **Born rule** tells us:

$$P(\text{outcome } k) = |\langle k | \psi \rangle|^2$$

If the qubit is already in a basis state, measurement is deterministic:
- Measure $|0\rangle$ $\rightarrow$ always 0
- Measure $|1\rangle$ $\rightarrow$ always 1

Let's verify this with 1000 shots.

In [2]:
%%
// Measure |0> -- should always give 0
c0, _ := builder.New("measure-zero", 1).MeasureAll().Build()
fmt.Println("Circuit: measure |0>")
gonbui.DisplayHTML(draw.SVG(c0))

sim := statevector.New(1)
counts0, _ := sim.Run(c0, 1000)
fmt.Printf("|0> measurement results: %v\n\n", counts0)

// Measure |1> -- apply X first, should always give 1
c1, _ := builder.New("measure-one", 1).X(0).MeasureAll().Build()
fmt.Println("Circuit: measure |1>")
gonbui.DisplayHTML(draw.SVG(c1))

counts1, _ := sim.Run(c1, 1000)
fmt.Printf("|1> measurement results: %v\n", counts1)

Circuit: measure |0>
|0> measurement results: map[0:1000]

Circuit: measure |1>
|1> measurement results: map[1:1000]


q0 
 
 M

q0 
 
 X 
 M

## Measuring Superposition

Things get interesting when we measure a qubit in superposition. Applying a Hadamard to $|0\rangle$ creates:

$$H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) = |+\rangle$$

The Born rule gives us $P(0) = |\frac{1}{\sqrt{2}}|^2 = \frac{1}{2}$ and $P(1) = \frac{1}{2}$. We expect roughly 50/50 results.

In [3]:
%%
// H|0> then measure -- should be ~50/50
c, _ := builder.New("superposition", 1).H(0).MeasureAll().Build()
gonbui.DisplayHTML(draw.SVG(c))

sim := statevector.New(1)
counts, _ := sim.Run(c, 1000)
fmt.Printf("Measurement results: %v\n", counts)
fmt.Printf("P(0) = %.3f, P(1) = %.3f\n",
	float64(counts["0"])/1000.0,
	float64(counts["1"])/1000.0)

// Visualize as a histogram
gonbui.DisplayHTML(viz.Histogram(counts))

q0 
 
 H 
 M

0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 0 
 
 1

Measurement results: map[0:502 1:498]
P(0) = 0.502, P(1) = 0.498


## Measurement Collapse: Measurement Destroys Superposition

A common misconception is that measurement simply "reads" the state without changing it. In reality, **measurement collapses the quantum state** into the observed basis state. After measurement, the qubit is no longer in superposition -- it is definitively $|0\rangle$ or $|1\rangle$.

We can prove this using a **mid-circuit measurement**. Consider this sequence:

1. $H|0\rangle \rightarrow |+\rangle$ (create superposition)
2. **Measure** $\rightarrow$ collapses to $|0\rangle$ or $|1\rangle$
3. $H$ again (apply Hadamard to the collapsed state)
4. **Measure** again

If measurement didn't collapse the state, the second H would undo the first (since $HH = I$), and the second measurement would always give 0. But because measurement **destroys** the superposition, the second measurement's outcome depends on the first measurement's result:

- If the first measurement gave 0: state is $|0\rangle$, then $H|0\rangle = |+\rangle$, second measurement is 50/50
- If the first measurement gave 1: state is $|1\rangle$, then $H|1\rangle = |-\rangle$, second measurement is also 50/50

We use `If` to apply a classically conditioned gate and `RunDynamic` for mid-circuit measurement support.

In [4]:
%%
// Build a dynamic circuit: H -> Measure -> H -> Measure
// Uses 2 classical bits to store both measurement results
c, _ := builder.New("collapse", 1).WithClbits(2).
	H(0).           // Step 1: create |+>
	Measure(0, 0).  // Step 2: measure -> collapses to |0> or |1>
	H(0).           // Step 3: H on collapsed state
	Measure(0, 1).  // Step 4: measure again
	Build()

fmt.Println("Circuit (dynamic):")
gonbui.DisplayHTML(draw.SVG(c))
fmt.Printf("Is dynamic: %v\n\n", c.IsDynamic())

sim := statevector.New(1)
counts, _ := sim.RunDynamic(c, 2000)
fmt.Println("Results (bit string = [c1 c0]):")
for bs, n := range counts {
	fmt.Printf("  %s: %d (%.1f%%)\n", bs, n, float64(n)/2000.0*100)
}

fmt.Println("\nKey insight: all four outcomes (00, 01, 10, 11) appear.")
fmt.Println("If measurement didn't collapse the state, HH=I would make")
fmt.Println("the second measurement always match the first.")
fmt.Println("The presence of mismatched results (01, 10) proves collapse!")

Circuit (dynamic):
Is dynamic: true



q0 
 
 H 
 M 
 H 
 M

Results (bit string = [c1 c0]):
  00: 526 (26.3%)
  11: 505 (25.2%)
  10: 480 (24.0%)
  01: 489 (24.4%)

Key insight: all four outcomes (00, 01, 10, 11) appear.
If measurement didn't collapse the state, HH=I would make
the second measurement always match the first.
The presence of mismatched results (01, 10) proves collapse!


## Partial Measurement on Entangled States

When qubits are entangled, measuring one instantly determines the other. Consider the Bell state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

If we measure only qubit 0:
- If we get 0, qubit 1 collapses to $|0\rangle$
- If we get 1, qubit 1 collapses to $|1\rangle$

The qubits are perfectly correlated. Let's demonstrate by measuring qubit 0 first, then qubit 1 separately.

In [5]:
%%
// Bell state: measure only qubit 0 first, then qubit 1
c, _ := builder.New("partial-measure", 2).WithClbits(2).
	H(0).CNOT(0, 1).  // Create Bell state |00> + |11>
	Measure(0, 0).     // Measure only qubit 0
	Measure(1, 1).     // Then measure qubit 1
	Build()

gonbui.DisplayHTML(draw.SVG(c))

sim := statevector.New(2)
counts, _ := sim.RunDynamic(c, 1000)
fmt.Println("Partial measurement results:")
for bs, n := range counts {
	fmt.Printf("  %s: %d\n", bs, n)
}
fmt.Println("\nOnly 00 and 11 appear -- measuring qubit 0 collapses qubit 1 too!")

gonbui.DisplayHTML(viz.Histogram(counts))

q0 
 
 q1 
 
 H 
 
 
 
 M 
 M

Partial measurement results:
  11: 492
  00: 508

Only 00 and 11 appear -- measuring qubit 0 collapses qubit 1 too!


0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 00 
 
 11

## Predict, Then Verify

**Question:** What happens if you measure a qubit **twice in a row** in the computational basis? Will the second measurement always agree with the first, or can it differ?

**Pause and predict** before running the next cell.

*Hint: Think about what the state looks like immediately after the first measurement.*

In [6]:
%%
// Start in superposition, then measure twice with no gates in between
c, _ := builder.New("double-measure", 1).WithClbits(2).
	H(0).           // Create superposition
	Measure(0, 0).  // First measurement -> collapses to |0> or |1>
	Measure(0, 1).  // Second measurement -> should match the first
	Build()

gonbui.DisplayHTML(draw.SVG(c))

sim := statevector.New(1)
counts, _ := sim.RunDynamic(c, 1000)
fmt.Println("Double measurement results (bit string = [c1 c0]):")
for bs, n := range counts {
	fmt.Printf("  %s: %d\n", bs, n)
}
fmt.Println("\nOnly 00 and 11 appear -- both measurements always agree.")
fmt.Println("The second measurement is completely determined by the first.")
fmt.Println("This confirms: once collapsed, the state stays collapsed.")

q0 
 
 H 
 M 
 M

Double measurement results (bit string = [c1 c0]):
  11: 500
  00: 500

Only 00 and 11 appear -- both measurements always agree.
The second measurement is completely determined by the first.
This confirms: once collapsed, the state stays collapsed.


## The No-Cloning Theorem

The **no-cloning theorem** states that it is impossible to create an identical copy of an arbitrary unknown quantum state. There is no unitary operation $U$ such that:

$$U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$$

for all $|\psi\rangle$.

A naive attempt might be to use a CNOT gate as a "copier." CNOT works for basis states:
- $\text{CNOT}|0\rangle|0\rangle = |0\rangle|0\rangle$ (copies $|0\rangle$)
- $\text{CNOT}|1\rangle|0\rangle = |1\rangle|1\rangle$ (copies $|1\rangle$)

But what about a superposition $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$?

$$\text{CNOT}|+\rangle|0\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

This is a Bell state -- an **entangled** state, not the product state $|+\rangle|+\rangle = \frac{1}{2}(|00\rangle + |01\rangle + |10\rangle + |11\rangle)$. The CNOT didn't copy the superposition; it created entanglement instead.

In [7]:
%%
// Attempt to "clone" |+> using CNOT
c, _ := builder.New("clone-attempt", 2).
	H(0).          // Prepare |+> on qubit 0
	CNOT(0, 1).    // "Copy" to qubit 1?
	MeasureAll().
	Build()

fmt.Println("Attempting to clone |+> with CNOT:")
gonbui.DisplayHTML(draw.SVG(c))

sim := statevector.New(2)

// First, look at the statevector before measurement
cNoMeas, _ := builder.New("clone-sv", 2).H(0).CNOT(0, 1).Build()
sim.Evolve(cNoMeas)
sv := sim.StateVector()
fmt.Println("Statevector after CNOT|+>|0>:")
for i, amp := range sv {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%02b> : %.4f (prob = %.2f)\n", i, amp, p)
	}
}

fmt.Println("\nIf cloning worked, we'd see |00>, |01>, |10>, |11> each with prob 0.25")
fmt.Println("Instead we see only |00> and |11> -- this is a Bell state, not a clone!")

// Run measurement to confirm
counts, _ := sim.Run(c, 1000)
fmt.Printf("\nMeasurement counts: %v\n", counts)
fmt.Println("Correlations prove entanglement, not independent copies.")

gonbui.DisplayHTML(viz.Histogram(counts))

Attempting to clone |+> with CNOT:
Statevector after CNOT|+>|0>:
  |00> : (0.7071+0.0000i) (prob = 0.50)
  |11> : (0.7071+0.0000i) (prob = 0.50)

If cloning worked, we'd see |00>, |01>, |10>, |11> each with prob 0.25
Instead we see only |00> and |11> -- this is a Bell state, not a clone!

Measurement counts: map[00:493 11:507]
Correlations prove entanglement, not independent copies.


q0 
 
 q1 
 
 H 
 
 
 
 M 
 M

0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 00 
 
 11

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. If you measure a qubit in state |+⟩ and get 0, what state is the qubit in afterward?
2. Why can't you use CNOT to copy an unknown quantum state?
3. True or false: Measuring a qubit twice in a row (with no gates in between) can give different results. Why?

## Exercises

### Exercise 1: Quantum Coin Flip

Design a "quantum coin flip" circuit -- a single qubit that produces a truly random 50/50 outcome. Run it for 1000 shots and display the histogram. This is the simplest quantum random number generator.

In [8]:
%%
// Exercise 1: Quantum coin flip
// Build the simplest quantum random number generator:
// a single qubit that produces a truly random 50/50 outcome.
// Expected: approximately 500 "0" and 500 "1" (±50)
//
// TODO: Apply the right gate to |0> to create equal superposition, then measure
// coin, _ := builder.New("coin-flip", 1). /* gate */ .MeasureAll().Build()
// fmt.Println(draw.String(coin))
//
// sim := statevector.New(1)
// flips, _ := sim.Run(coin, 1000)
// fmt.Printf("Heads (0): %d, Tails (1): %d\n", flips["0"], flips["1"])
//
// gonbui.DisplayHTML(viz.Histogram(flips))
fmt.Println("Uncomment the code above and fill in the gate!")

Uncomment the code above and fill in the gate!


### Exercise 2: No-Cloning Verification

Show that cloning is impossible: use CNOT to try to "copy" $|+\rangle$ to a second qubit, then measure both. If cloning worked, both qubits would be independent and you'd see all four outcomes (00, 01, 10, 11) with equal probability. Instead, you should see only correlated outcomes.

In [9]:
%%
// Exercise 2: Verify the no-cloning theorem
// Use CNOT to try to "copy" |+> to a second qubit, then measure both.
// If cloning worked, you'd see all four outcomes (00, 01, 10, 11) equally.
// Instead, you should see only correlated outcomes (00 and 11).
// Expected: ~500 "00" and ~500 "11", with 0 counts for "01" and "10"
//
// TODO: Build the "cloning" circuit and an independent-copies circuit for comparison
// clone, _ := builder.New("no-clone", 2).
// 	H(0).           // prepare |+> on qubit 0
// 	CNOT(0, 1).     // attempt to copy
// 	MeasureAll().
// 	Build()
//
// sim := statevector.New(2)
// results, _ := sim.Run(clone, 1000)
//
// fmt.Println("CNOT 'cloning' results:")
// for bs, n := range results {
// 	fmt.Printf("  %s: %d (%.1f%%)\n", bs, n, float64(n)/1000.0*100)
// }
//
// // Check: are 01 and 10 present?
// if results["01"] == 0 && results["10"] == 0 {
// 	fmt.Println("\nNo 01 or 10 outcomes -- qubits are perfectly correlated.")
// 	fmt.Println("This is ENTANGLEMENT, not cloning. No-cloning theorem confirmed!")
// }
//
// // Now build two truly independent |+> qubits for comparison
// indep, _ := builder.New("independent", 2).
// 	H(0).H(1).     // independent |+> on each qubit
// 	MeasureAll().
// 	Build()
//
// sim2 := statevector.New(2)
// indepResults, _ := sim2.Run(indep, 1000)
// fmt.Println("\nFor comparison, two independent |+> qubits (no CNOT):")
// for bs, n := range indepResults {
// 	fmt.Printf("  %s: %d (%.1f%%)\n", bs, n, float64(n)/1000.0*100)
// }
// fmt.Println("All four outcomes appear -- these are truly independent.")
//
// gonbui.DisplayHTML(viz.Histogram(results))
fmt.Println("Uncomment the code above and complete the no-cloning experiment!")

Uncomment the code above and complete the no-cloning experiment!


## Key Takeaways

1. **Born rule**: The probability of measuring outcome $k$ is $|\langle k|\psi\rangle|^2$. Basis states give deterministic results; superpositions give probabilistic results.

2. **Measurement collapse**: Measurement is not a passive observation. It irreversibly destroys superposition, projecting the state onto the measured basis state. This is why quantum information is fundamentally different from classical information.

3. **Repeated measurement is idempotent**: Measuring the same qubit twice without intervening gates always gives the same result. Once collapsed, the state stays collapsed.

4. **Partial measurement and entanglement**: Measuring one qubit of an entangled pair instantaneously determines the other. This is the basis for quantum teleportation and superdense coding.

5. **No-cloning theorem**: You cannot copy an unknown quantum state. CNOT appears to copy basis states but creates entanglement (not copies) when applied to superpositions. This is fundamental to quantum cryptography -- an eavesdropper cannot copy qubits without disturbing them.

6. **Mid-circuit measurement**: Goqu supports dynamic circuits where measurements can occur in the middle of a circuit. The results can be used to conditionally apply gates via `If` and `IfBlock`, enabling feed-forward quantum computation.